# Exercício Regressão Linear com PyTorch

O objetivo desse notebook é reescrevê-lo porém utilizando tensores do PyTorch.

Os nomes das funções do PyTorch são próximas das funções do Torch original escrito
na linguagem Lua, porém não são iguais.

## Importação dos pacotes

In [ ]:
%matplotlib inline
import torch

import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

torch.manual_seed(1234)

def print_assert(r):
    if r:
        return 'OK!'
    else:
        return 'Incorreto!'

## Dataset

In [ ]:
import pandas as pd
iris = load_iris()
data = iris.data[iris.target==1,::2]

x_in = data[:,0:1]
y_in = data[:,1:2]

x_train = torch.FloatTensor(x_in)
y_train = torch.FloatTensor(y_in)

n_samples = x_train.size(0)
print('x_train.shape:',x_train.shape, type(x_train))
print('y_train.shape:',y_train.shape, type(y_train))

### Normalização dos dados

In [ ]:
x_train -= x_train.min()
x_train /= x_train.max()
y_train -= y_train.min()
y_train /= y_train.max()

plt.scatter(x_train, y_train)
plt.xlabel('Comprimento sepalas')
plt.ylabel('Comprimento petalas')

### Exercício 1 - Criação do `x_train_bias`

In [ ]:
# SOLUÇÃO - Exercício 1
#
# torch.ones(n_samples, 1) cria um tensor coluna de 1s com shape (50, 1)
# torch.cat([...], dim=1) concatena os dois tensores lado a lado (coluna a coluna)
# Resultado: shape (50, 2) — coluna de 1s (bias) + coluna de x_train

x_train_bias = torch.cat([torch.ones(n_samples, 1), x_train], dim=1)

print('x_train_bias[:5]:')
print(x_train_bias[:5])

In [ ]:
# Verificação do exercício
print('x_train_bias é um tensor: {}'.format(print_assert(
    isinstance(x_train_bias, torch.FloatTensor)
)))
print('tamanho do x_train_bias: {}'.format(print_assert(
    x_train_bias.size() == torch.Size([50, 2])
)))
print('primeira coluna é só de uns: {}'.format(print_assert(
    (x_train_bias[:, 0] - torch.ones(n_samples)).sum() == 0
)))
print('segunda coluna é igual a x_train: {}'.format(print_assert(
    (x_train_bias[:, 1] - x_train.reshape(50)).sum() == 0
)))

## Modelo da rede

In [ ]:
class Net():
    def __init__(self, n_in, n_out):
        self.w = torch.Tensor(n_out, n_in)
        self.w.uniform_(-0.1, 0.1)

    def forward(self, x_bias):
        return x_bias.matmul(torch.t(self.w))

model = Net(2, 1)

## Treinamento

### Exercício 2 - Treinamento com PyTorch

In [ ]:
# SOLUÇÃO - Exercício 2
#
# Tradução das operações NumPy para PyTorch:
#
#   NumPy                              PyTorch
#   np.square(a - b).mean()     -->    torch.mean((a - b) ** 2)
#   (X.T).dot(Y)                -->    X.t().matmul(Y)    ou    X.t().mm(Y)
#   X.dot(w.T)                  -->    X.matmul(torch.t(w))
#   w.T                         -->    torch.t(w)

model = Net(2, 1)  # reinicializa para garantir pesos aleatórios frescos
perdas = []

num_epochs = 100
learning_rate = 0.5

for epoch in range(num_epochs):

    # forward - predict
    y_pred = model.forward(x_train_bias)

    # loss: MSE com PyTorch
    loss = torch.mean((y_pred - y_train) ** 2)
    perdas.append(loss.item())  # .item() extrai o valor escalar do tensor

    # gradiente: (2/M) * X^T * (X*w^T - y)
    w_grad = (2.0 / n_samples) * x_train_bias.t().matmul(
        x_train_bias.matmul(torch.t(model.w)) - y_train
    )

    # gradiente descendente: w = w - lr * grad^T
    model.w = model.w - learning_rate * torch.t(w_grad)

    if (epoch+1) % 20 == 0:
        print('Epoch[{}/{}], loss: {:.6f}'.format(epoch+1, num_epochs, loss.item()))

In [ ]:
# Curva de aprendizado
plt.plot(perdas, color='r', marker='o', markersize=3)
plt.ylabel('Perda (MSE)')
plt.xlabel('Época')
plt.title('Convergência do Gradiente Descendente')
plt.show()

## Avaliação

In [ ]:
y_pred = model.forward(x_train_bias)
plt.plot(x_train.numpy(), y_train.numpy(), 'ro', label='Original data')
plt.plot(x_train.numpy(), y_pred.numpy(), 'kx-', label='Fitting Line')
plt.legend()
plt.show()

In [ ]:
print('Pesos treinados (GD):', model.w)

# Solução analítica
x_bias = x_train_bias
w_opt = (torch.inverse(x_bias.t().mm(x_bias)).mm(x_bias.t())).mm(y_train)
print('Pesos ótimos analíticos:', w_opt.t())

# Comparação das perdas
loss_gd = torch.mean((model.forward(x_train_bias) - y_train) ** 2).item()

model_temp = Net(2, 1)
model_temp.w = w_opt.t()
loss_opt = torch.mean((model_temp.forward(x_train_bias) - y_train) ** 2).item()

print(f'\nPerda - Gradiente Descendente:   {loss_gd:.6f}')
print(f'Perda - Solução Ótima Analítica: {loss_opt:.6f}')